In [1]:
# You must run the lines below at the start of every new Python session.
import numpy as np                    # Mathematical computing
import pandas as pd                   # Data manipulation
import statsmodels.api as sm          # Statistical models
import statsmodels.formula.api as smf # Formula-based regression
from linearmodels.iv import IV2SLS    # Two-stage least squares for IV
import matplotlib.pyplot as plt       # Visualization

In [4]:
data = pd.read_csv(r"data.csv")

In [16]:
data.head(2)

,member_id,received_email,enrolled_program,annual_spending
0,1,0,0,65.11
1,2,1,0,56.47


In [23]:
X_first = data[['received_email']]
X_first = sm.add_constant(X_first)
y_first = data['enrolled_program']

In [24]:
first_stage = sm.OLS(y_first, X_first).fit(cov_type='HC3')
print("\nFirst Stage Results:")
print(first_stage.summary())


First Stage Results:
                            OLS Regression Results                            
Dep. Variable:       enrolled_program   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     15.55
Date:                Tue, 25 Aug 2026   Prob (F-statistic):           8.24e-05
Time:                        15:13:14   Log-Likelihood:                -1990.7
No. Observations:                3000   AIC:                             3985.
Df Residuals:                    2998   BIC:                             3997.
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.2

A regressão indica que de fato há uma forte relação entre as variáveis já que o f-statistics ficou em 4565

In [25]:
# Extract F-statistic for weak instrument test
t_stat = first_stage.tvalues['received_email']
f_stat = t_stat**2
print(f"\nFirst-stage F-statistic: {f_stat:.2f}")
print("Rule of thumb: F > 10 indicates a strong instrument")



First-stage F-statistic: 15.55
Rule of thumb: F > 10 indicates a strong instrument


In [30]:
check = smf.ols(data = data, 
                formula = 'annual_spending ~ 1 + enrolled_program').fit(cov_type='HC3')

print(check.summary())

                            OLS Regression Results                            
Dep. Variable:        annual_spending   R-squared:                       0.303
Model:                            OLS   Adj. R-squared:                  0.302
Method:                 Least Squares   F-statistic:                     1325.
Date:                Tue, 25 Aug 2026   Prob (F-statistic):          1.55e-240
Time:                        15:24:59   Log-Likelihood:                -11810.
No. Observations:                3000   AIC:                         2.362e+04
Df Residuals:                    2998   BIC:                         2.364e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           52.5304      0.280  

In [26]:
# Qual o efeito do email sobre o gasto anual?
reduced_form =  smf.ols('annual_spending ~ received_email', data=data).fit(cov_type='HC3')

In [27]:
# 2SLS: efeito da matrícula no gasto, usando o e-mail como instrumento
mod = IV2SLS.from_formula(
    'annual_spending ~ 1 + [enrolled_program ~ received_email]', data=data
)
iv_result = mod.fit(cov_type='robust')
print(iv_result.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:        annual_spending   R-squared:                     -0.0032
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0036
No. Observations:                3000   F-statistic:                    0.0001
Date:                Tue, Aug 25 2026   P-value (F-stat)                0.9908
Time:                        15:19:52   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                                Parameter Estimates                                 
                  Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------
Intercept            46.736     2.6777     17.454     0.0000      41.488      51.984
enrolled_program     0.0920 